# 04 — Tokenization, Stopword Removal and Lemmatization

This notebook:

1. Loads the cleaned sentence dataset.
2. Tokenizes text with spaCy.
3. Removes stopwords while retaining sentiment-sensitive words.
4. Lemmatizes the remaining tokens.
5. Rejoins the tokens into `processed_text`.
6. Produces optional word clouds.
7. Saves a model-ready dataset.

> The pasted source code applied the spaCy preprocessing function to a Python list. Here it is applied correctly by joining the tokens before lemmatization.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "06_sentences_processed.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_text_labelled = pd.read_csv(INPUT_PATH)

print("Loaded:", INPUT_PATH)
print("Rows:", len(df_text_labelled))
display(df_text_labelled.head())

## Install and load spaCy

In [ ]:
%pip install -q spacy

In [ ]:
# Run once in a new environment if the model is not installed:
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise OSError(
        "spaCy model 'en_core_web_sm' is not installed. "
        "Run: !python -m spacy download en_core_web_sm"
    )

## Tokenization

In [ ]:
def tokenize_text(text):
    doc = nlp(str(text))
    return [token.text for token in doc]

df_text_labelled["tokens"] = (
    df_text_labelled["review_text"]
    .apply(tokenize_text)
)

display(
    df_text_labelled[
        ["review_text", "tokens"]
    ].head()
)

## Stopword removal

`not`, `no`, `never`, and `but` are retained because they can materially change sentiment interpretation.

In [ ]:
negation_words = {
    "no",
    "not",
    "never",
    "but"
}

def remove_stopwords_keep_negation(text):
    doc = nlp(str(text))

    return [
        token.text
        for token in doc
        if (
            not token.is_space
            and not token.is_punct
            and (
                not token.is_stop
                or token.text.lower() in negation_words
            )
        )
    ]

df_text_labelled["tokens_no_stopwords"] = (
    df_text_labelled["review_text"]
    .apply(remove_stopwords_keep_negation)
)

display(
    df_text_labelled[
        ["review_text", "tokens_no_stopwords"]
    ].head()
)

## Lemmatization

In [ ]:
def lemmatize_tokens(tokens):
    doc = nlp(" ".join(map(str, tokens)))

    return [
        token.lemma_.lower()
        for token in doc
        if (
            not token.is_space
            and not token.is_punct
        )
    ]

df_text_labelled["lemmas"] = (
    df_text_labelled["tokens_no_stopwords"]
    .apply(lemmatize_tokens)
)

display(
    df_text_labelled[
        ["review_text", "tokens_no_stopwords", "lemmas"]
    ].head()
)

## Convert tokens back to text

In [ ]:
df_text_labelled["processed_text"] = (
    df_text_labelled["lemmas"]
    .apply(lambda tokens: " ".join(tokens))
)

display(
    df_text_labelled[
        ["review_text", "lemmas", "processed_text"]
    ].head(10)
)

## Word clouds

In [ ]:
%pip install -q wordcloud matplotlib

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

all_text = " ".join(
    df_text_labelled["processed_text"]
    .dropna()
    .astype(str)
)

if all_text.strip():
    wc = WordCloud(
        width=1200,
        height=600,
        background_color="white",
        max_words=100,
        collocations=False
    ).generate(all_text)

    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Employee Reviews")
    plt.show()

for sentiment_label in ["Positive", "Negative"]:
    sentiment_text = " ".join(
        df_text_labelled.loc[
            df_text_labelled["sentiment"] == sentiment_label,
            "processed_text"
        ]
        .dropna()
        .astype(str)
    )

    if sentiment_text.strip():
        wc = WordCloud(
            width=1200,
            height=600,
            background_color="white",
            max_words=100,
            collocations=False
        ).generate(sentiment_text)

        plt.figure(figsize=(14, 7))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Word Cloud for {sentiment_label} Reviews")
        plt.show()

## Save model-ready dataset

In [ ]:
output_path = PROCESSED_DIR / "07_model_ready.csv"

df_text_labelled.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)